# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub scikit-learn

In [3]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

# Load the token safely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add it in the Colab Secrets panel."
    )

user_info = whoami(token=HF_TOKEN)
print("Hugging Face login successful:", user_info["name"])

# Connect DuckDB
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection ready.")
print("Development month: March 2026")
print("Feature window: March 1–15")
print("Outcome window: March 16–31")

Hugging Face login successful: zafar4050
DuckDB connection ready.
Development month: March 2026
Feature window: March 1–15
Outcome window: March 16–31


## 1. Build the Feature Vector

My unit of analysis is one anonymized content page for one client.

I use March 1–15, 2026 as the feature window. All model features are calculated only from data available during this period.

March 16–31 is used only to define the provisional declining proxy. Measurements from this outcome window are not included in the honest feature vector.

The raw daily data is aggregated into one page-level row.

### Numeric features

- `log_impressions_pre`
- `clicks_pre`
- `ctr_pre`
- `avg_position_pre`
- `active_days_pre`
- `position_volatility_pre`

### Categorical feature

- `position_band`

The position band is derived only from the feature-window average position and is one-hot encoded.

Missing numeric measurements are filled with the median of the development slice. Missing categorical values are assigned to an `unknown` category.

Client and content identifiers are retained only as context for grouping, validation, and review. They are not included as model features.

In [4]:
raw_feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature-window totals
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions_pre,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS clicks_pre,

        AVG(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS avg_position_pre,

        STDDEV_SAMP(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS position_volatility_pre,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                 AND COALESCE(gsc_impressions, 0) > 0
                THEN report_date
            END
        ) AS active_days_pre,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS feature_available_days,

        -- Outcome-window information: used only for target creation
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS outcome_available_days

    FROM {MARCH_FACT}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Raw page rows:", f"{len(raw_feature_frame):,}")
raw_feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw page rows: 331,437


,client_hash_id,content_hash_id,impressions_pre,clicks_pre,avg_position_pre,position_volatility_pre,active_days_pre,feature_available_days,outcome_impressions,outcome_available_days
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.659683,2.311228,15,15,20.0,9
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,4.086084,0.560314,13,13,403.0,16
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,4.449176,1.003601,13,13,343.0,16
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,6.600595,3.529207,14,14,26.0,13
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,1.883472,0.777534,14,14,1087.0,16


In [9]:
# Keep pages with enough visibility and enough measured days
eligible = raw_feature_frame[
    (raw_feature_frame["impressions_pre"] >= 100)
    & (raw_feature_frame["feature_available_days"] >= 5)
    & (raw_feature_frame["outcome_available_days"] >= 5)
].copy()

# Average daily impressions in each window
eligible["feature_daily_impressions"] = (
    eligible["impressions_pre"]
    / eligible["feature_available_days"]
)

eligible["outcome_daily_impressions"] = (
    eligible["outcome_impressions"]
    / eligible["outcome_available_days"]
)

# Provisional future-window proxy
eligible["is_declining_proxy"] = (
    eligible["outcome_daily_impressions"]
    < 0.80 * eligible["feature_daily_impressions"]
).astype(int)

# Engineered feature: log transform reduces extreme scale differences
eligible["log_impressions_pre"] = np.log1p(
    eligible["impressions_pre"]
)

# Engineered feature: safe CTR
eligible["ctr_pre"] = np.where(
    eligible["impressions_pre"] > 0,
    eligible["clicks_pre"] / eligible["impressions_pre"],
    np.nan,
)

# Categorical feature derived from pre-outcome position
eligible["position_band"] = pd.cut(
    eligible["avg_position_pre"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=[
        "top_3",
        "page_1",
        "page_2",
        "deep",
    ],
)

eligible["position_band"] = (
    eligible["position_band"]
    .astype("object")
    .fillna("unknown")
)

print("Eligible page rows:", f"{len(eligible):,}")
print(
    "Declining proxy rate:",
    round(eligible["is_declining_proxy"].mean(), 3),
)

eligible[
    [
        "client_hash_id",
        "content_hash_id",
        "log_impressions_pre",
        "clicks_pre",
        "ctr_pre",
        "avg_position_pre",
        "active_days_pre",
        "position_volatility_pre",
        "position_band",
        "is_declining_proxy",
    ]
].head()

Eligible page rows: 76,201
Declining proxy rate: 0.345


,client_hash_id,content_hash_id,log_impressions_pre,clicks_pre,ctr_pre,avg_position_pre,active_days_pre,position_volatility_pre,position_band,is_declining_proxy
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,5.298317,2.0,0.010050,4.086084,13,0.560314,page_1,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,6.148468,1.0,0.002141,4.449176,13,1.003601,page_1,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,6.648985,1.0,0.001297,1.883472,14,0.777534,top_3,0
6,client_62f4a7e64f5e0096,content_92c381fbd361212e,5.455321,0.0,0.000000,4.807270,13,1.887822,page_1,0
7,client_62f4a7e64f5e0096,content_97188a7032a705cf,5.442418,2.0,0.008696,3.760644,13,1.622677,page_1,0


In [10]:
numeric_features = [
    "log_impressions_pre",
    "clicks_pre",
    "ctr_pre",
    "avg_position_pre",
    "active_days_pre",
    "position_volatility_pre",
]

categorical_features = [
    "position_band",
]

print("Missing values before filling:")
print(eligible[numeric_features + categorical_features].isna().sum())

Missing values before filling:
log_impressions_pre        0
clicks_pre                 0
ctr_pre                    0
avg_position_pre           0
active_days_pre            0
position_volatility_pre    0
position_band              0
dtype: int64


In [11]:
processed = eligible.copy()

# Median fill for numeric features
numeric_fill_values = {}

for column in numeric_features:
    fill_value = processed[column].median()

    if pd.isna(fill_value):
        fill_value = 0.0

    numeric_fill_values[column] = float(fill_value)
    processed[column] = processed[column].fillna(fill_value)

# Unknown category for missing categorical values
for column in categorical_features:
    processed[column] = (
        processed[column]
        .astype("object")
        .fillna("unknown")
    )

# One-hot encode categorical feature
encoded_position = pd.get_dummies(
    processed["position_band"],
    prefix="position_band",
    dtype=int,
)

# Build final model matrix
X_final = pd.concat(
    [
        processed[numeric_features].reset_index(drop=True),
        encoded_position.reset_index(drop=True),
    ],
    axis=1,
)

# Context is kept separately
context_frame = processed[
    [
        "client_hash_id",
        "content_hash_id",
    ]
].reset_index(drop=True)

y_final = processed[
    "is_declining_proxy"
].reset_index(drop=True)

final_feature_columns = X_final.columns.tolist()

print("Final feature-vector shape:", X_final.shape)
print("Final feature columns:")
print(final_feature_columns)
print("Total missing values:", int(X_final.isna().sum().sum()))

assert X_final.isna().sum().sum() == 0

X_final.head()

Final feature-vector shape: (76201, 10)
Final feature columns:
['log_impressions_pre', 'clicks_pre', 'ctr_pre', 'avg_position_pre', 'active_days_pre', 'position_volatility_pre', 'position_band_deep', 'position_band_page_1', 'position_band_page_2', 'position_band_top_3']
Total missing values: 0


,log_impressions_pre,clicks_pre,ctr_pre,avg_position_pre,active_days_pre,position_volatility_pre,position_band_deep,position_band_page_1,position_band_page_2,position_band_top_3
0,5.298317,2.0,0.010050,4.086084,13,0.560314,0,1,0,0
1,6.148468,1.0,0.002141,4.449176,13,1.003601,0,1,0,0
2,6.648985,1.0,0.001297,1.883472,14,0.777534,0,0,0,1
3,5.455321,0.0,0.000000,4.807270,13,1.887822,0,1,0,0
4,5.442418,2.0,0.008696,3.760644,13,1.622677,0,1,0,0


## 2. Feature Notes

### `log_impressions_pre`

Meaning: The natural logarithm of total GSC impressions during March 1–15.

Missing handling: The raw impressions value is aggregated with zero-safe logic. Any remaining missing value is filled with the development-slice median.

Categorical: No.

Available when: Yes. It uses only impressions measured before March 16.

---

### `clicks_pre`

Meaning: Total GSC clicks during March 1–15.

Missing handling: Missing values are filled with the median.

Categorical: No.

Available when: Yes. It is observed during the feature window.

---

### `ctr_pre`

Meaning: Feature-window clicks divided by feature-window impressions.

Missing handling: Undefined values are replaced with the median after safe division.

Categorical: No.

Available when: Yes. Both clicks and impressions are observed before the outcome window.

---

### `avg_position_pre`

Meaning: The page's average GSC search position during March 1–15.

Missing handling: Missing values are filled with the median.

Categorical: No.

Available when: Yes. It is calculated only from feature-window measurements.

---

### `active_days_pre`

Meaning: The number of feature-window days on which the page received at least one impression.

Missing handling: The SQL aggregation returns zero when no active days are found. Any unexpected missing value is filled with the median.

Categorical: No.

Available when: Yes. It counts only days before the prediction moment.

---

### `position_volatility_pre`

Meaning: The standard deviation of the page's daily average search position during March 1–15.

Missing handling: A page may have too few position observations to calculate a standard deviation. Missing values are filled with the median.

Categorical: No.

Available when: Yes. It uses only position observations from the feature window.

---

### `position_band`

Meaning: A categorical grouping of feature-window average position: top 3, page 1, page 2, deep, or unknown.

Missing handling: Missing values are mapped to `unknown`.

Categorical handling: The categories are converted into one-hot encoded binary columns.

Available when: Yes. The band is derived only from pre-outcome average position.

In [12]:
feature_notes = pd.DataFrame(
    [
        {
            "feature": "log_impressions_pre",
            "meaning": "Log of feature-window impressions",
            "missing_handling": "Median fill",
            "categorical": "No",
            "available_before_prediction": "Yes",
        },
        {
            "feature": "clicks_pre",
            "meaning": "Total feature-window clicks",
            "missing_handling": "Median fill",
            "categorical": "No",
            "available_before_prediction": "Yes",
        },
        {
            "feature": "ctr_pre",
            "meaning": "Clicks divided by impressions",
            "missing_handling": "Safe division and median fill",
            "categorical": "No",
            "available_before_prediction": "Yes",
        },
        {
            "feature": "avg_position_pre",
            "meaning": "Average feature-window position",
            "missing_handling": "Median fill",
            "categorical": "No",
            "available_before_prediction": "Yes",
        },
        {
            "feature": "active_days_pre",
            "meaning": "Days with positive impressions",
            "missing_handling": "Median fill",
            "categorical": "No",
            "available_before_prediction": "Yes",
        },
        {
            "feature": "position_volatility_pre",
            "meaning": "Variation in daily search position",
            "missing_handling": "Median fill",
            "categorical": "No",
            "available_before_prediction": "Yes",
        },
        {
            "feature": "position_band",
            "meaning": "Bucketed feature-window position",
            "missing_handling": "Unknown category",
            "categorical": "One-hot encoded",
            "available_before_prediction": "Yes",
        },
    ]
)

feature_notes

,feature,meaning,missing_handling,categorical,available_before_prediction
0,log_impressions_pre,Log of feature-window impressions,Median fill,No,Yes
1,clicks_pre,Total feature-window clicks,Median fill,No,Yes
2,ctr_pre,Clicks divided by impressions,Safe division and median fill,No,Yes
3,avg_position_pre,Average feature-window position,Median fill,No,Yes
4,active_days_pre,Days with positive impressions,Median fill,No,Yes
5,position_volatility_pre,Variation in daily search position,Median fill,No,Yes
6,position_band,Bucketed feature-window position,Unknown category,One-hot encoded,Yes


In [13]:
missing_receipt = pd.DataFrame(
    {
        "feature": numeric_features,
        "fill_value": [
            numeric_fill_values[column]
            for column in numeric_features
        ],
        "missing_after_fill": [
            int(processed[column].isna().sum())
            for column in numeric_features
        ],
    }
)

missing_receipt

,feature,fill_value,missing_after_fill
0,log_impressions_pre,6.381816,0
1,clicks_pre,1.000000,0
2,ctr_pre,0.001346,0
3,avg_position_pre,7.588764,0
4,active_days_pre,15.000000,0
5,position_volatility_pre,3.403126,0


## 3. The Leakage Hunt

I tested the final feature vector for four main risks.

### Label-derived fields

Fields such as `decline_ratio`, `outcome_daily_impressions`, and the target itself would directly or indirectly reveal the answer.

### Future-window fields

Any measurement from March 16–31 occurs after the prediction moment and must not be used as an honest feature.

### Product decisions and flags

Fields such as `health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, or refresh flags would create a circular result if they were used as model features.

### Private or identifier fields

Raw URLs, domains, search queries, titles, client names, credentials, and private text are not used.

Anonymized client and content identifiers are kept only outside the model matrix for grouping and review. Their scrambled values are not model features.

In [14]:
forbidden_exact_fields = {
    "is_declining_proxy",
    "outcome_impressions",
    "outcome_daily_impressions",
    "outcome_available_days",
    "decline_ratio",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "refresh_flag",
}

forbidden_name_patterns = [
    "outcome",
    "future",
    "label",
    "decline_ratio",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_flag",
]

privacy_patterns = [
    "url",
    "domain",
    "raw_query",
    "query_text",
    "title",
    "client_name",
    "credential",
    "token",
    "email",
]

feature_name_set = set(final_feature_columns)

exact_leaks = sorted(
    feature_name_set.intersection(forbidden_exact_fields)
)

pattern_leaks = sorted(
    feature
    for feature in final_feature_columns
    if any(
        pattern in feature.lower()
        for pattern in forbidden_name_patterns
    )
)

privacy_risks = sorted(
    feature
    for feature in final_feature_columns
    if any(
        pattern in feature.lower()
        for pattern in privacy_patterns
    )
)

identifier_leaks = sorted(
    set(final_feature_columns).intersection(
        {
            "client_hash_id",
            "content_hash_id",
        }
    )
)

audit_results = pd.DataFrame(
    [
        {
            "check": "Exact forbidden fields",
            "matches": exact_leaks,
            "passed": len(exact_leaks) == 0,
        },
        {
            "check": "Forbidden name patterns",
            "matches": pattern_leaks,
            "passed": len(pattern_leaks) == 0,
        },
        {
            "check": "Privacy-sensitive patterns",
            "matches": privacy_risks,
            "passed": len(privacy_risks) == 0,
        },
        {
            "check": "Identifiers inside model matrix",
            "matches": identifier_leaks,
            "passed": len(identifier_leaks) == 0,
        },
    ]
)

audit_results

,check,matches,passed
0,Exact forbidden fields,[],True
1,Forbidden name patterns,[],True
2,Privacy-sensitive patterns,[],True
3,Identifiers inside model matrix,[],True


In [15]:
assert len(exact_leaks) == 0
assert len(pattern_leaks) == 0
assert len(privacy_risks) == 0
assert len(identifier_leaks) == 0

print("Leakage and privacy audit passed.")
print("No future-window, label-derived, product-decision, or private fields")
print("are present in the honest feature matrix.")

Leakage and privacy audit passed.
No future-window, label-derived, product-decision, or private fields
are present in the honest feature matrix.


### Deliberate Leakage Test

To test the audit, I deliberately create `decline_ratio`.

This ratio uses outcome-window impressions and directly determines the declining proxy. It therefore contains the answer in disguised form.

I compare an honest model against a leaky model using the same client-grouped split.

The leaky score is expected to become unrealistically high. After demonstrating the problem, the leaking field is deleted and is not retained in the final feature vector.

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score

groups = context_frame["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(
        X_final,
        y_final,
        groups=groups,
    )
)

X_train = X_final.iloc[train_index]
X_test = X_final.iloc[test_index]

y_train = y_final.iloc[train_index]
y_test = y_final.iloc[test_index]

train_clients = set(groups.iloc[train_index])
test_clients = set(groups.iloc[test_index])

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients)),
)

Train clients: 27
Test clients: 9
Client overlap: 0


In [17]:
honest_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

honest_model.fit(X_train, y_train)

honest_probability = honest_model.predict_proba(
    X_test
)[:, 1]

honest_prediction = honest_model.predict(X_test)

honest_auc = roc_auc_score(
    y_test,
    honest_probability,
)

honest_accuracy = accuracy_score(
    y_test,
    honest_prediction,
)

print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Honest ROC AUC: {honest_auc:.3f}")

Honest accuracy: 0.528
Honest ROC AUC: 0.568


In [18]:
# Deliberately create the leaking field
decline_ratio = (
    processed["outcome_daily_impressions"]
    / processed["feature_daily_impressions"]
).replace([np.inf, -np.inf], np.nan)

decline_ratio = decline_ratio.fillna(
    decline_ratio.median()
).reset_index(drop=True)

X_leaky = X_final.copy()
X_leaky["decline_ratio"] = decline_ratio

leaky_train = X_leaky.iloc[train_index]
leaky_test = X_leaky.iloc[test_index]

leaky_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    random_state=42,
    n_jobs=-1,
)

leaky_model.fit(leaky_train, y_train)

leaky_probability = leaky_model.predict_proba(
    leaky_test
)[:, 1]

leaky_prediction = leaky_model.predict(
    leaky_test
)

leaky_auc = roc_auc_score(
    y_test,
    leaky_probability,
)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_prediction,
)

print("DELIBERATE LEAKAGE TEST")
print("-" * 45)
print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Leaky accuracy:  {leaky_accuracy:.3f}")
print(f"Honest ROC AUC:  {honest_auc:.3f}")
print(f"Leaky ROC AUC:   {leaky_auc:.3f}")

DELIBERATE LEAKAGE TEST
---------------------------------------------
Honest accuracy: 0.528
Leaky accuracy:  1.000
Honest ROC AUC:  0.568
Leaky ROC AUC:   1.000


In [22]:
majority_baseline = max(
    y_test.mean(),
    1 - y_test.mean(),
)

print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")
print(f"Honest model accuracy: {honest_accuracy:.3f}")
print(f"Honest ROC AUC: {honest_auc:.3f}")

Majority-class baseline accuracy: 0.647
Honest model accuracy: 0.528
Honest ROC AUC: 0.568


In [19]:
# Permanently remove the leaking field
X_leaky.drop(
    columns=["decline_ratio"],
    inplace=True,
)

assert "decline_ratio" not in X_leaky.columns
assert list(X_leaky.columns) == list(X_final.columns)

print("decline_ratio removed.")
print("Final honest vector retained.")
print("Final feature count:", X_final.shape[1])
print(f"Retained honest ROC AUC: {honest_auc:.3f}")

decline_ratio removed.
Final honest vector retained.
Final feature count: 10
Retained honest ROC AUC: 0.568


### Leakage Test Result

The honest grouped model achieved a ROC AUC of **[insert honest AUC]**.

After I deliberately added `decline_ratio`, the score increased to **[insert leaky AUC]**.

This increase did not represent improved generalization. The leaking feature used outcome-window information and directly encoded the threshold used to create the target.

I removed `decline_ratio` and retained the honest feature vector.

The final feature vector contains only measurements that were available before March 16, 2026.

## 4. What I Excluded and Why

### `outcome_impressions`

Excluded because it is measured after the prediction moment.

### `outcome_daily_impressions`

Excluded because it uses the future outcome window.

### `outcome_available_days`

Excluded from the model because it describes the target period rather than the feature period.

### `decline_ratio`

Excluded because the proxy label is directly defined from this ratio. Including it would reveal the answer.

### `is_declining_proxy`

Used only as the target. It is never included in the model feature matrix.

### `client_hash_id`

Used only for GroupShuffleSplit and aggregated reporting. The scrambled identifier itself has no predictive meaning.

### `content_hash_id`

Used only to identify the anonymized page and connect recommendations to rows. It is not a model feature.

### `report_date`

Used to define the feature and outcome windows. It is not directly included as a predictive feature.

### Product scores and flags

Fields such as health scores, priority scores, refresh flags, CTR-fix flags, and action labels are excluded because they represent previous product decisions. Using them would create circular results.

### Raw URLs, domains, queries, titles, and client names

Excluded for privacy and public-safety reasons. These fields are not required for this analysis.

### Credentials and tokens

Never stored in the notebook or repository. The Hugging Face token is loaded from Colab Secrets.

### June 2026 sample

Excluded from development because June is reserved as a sealed later test month.

In [20]:
excluded_fields = pd.DataFrame(
    [
        {
            "field": "outcome_impressions",
            "reason": "Measured after the decision moment",
        },
        {
            "field": "outcome_daily_impressions",
            "reason": "Future-window measurement",
        },
        {
            "field": "outcome_available_days",
            "reason": "Target-window context, not a feature",
        },
        {
            "field": "decline_ratio",
            "reason": "Directly defines the proxy label",
        },
        {
            "field": "is_declining_proxy",
            "reason": "Target only, never a model feature",
        },
        {
            "field": "client_hash_id",
            "reason": "Grouping and validation only",
        },
        {
            "field": "content_hash_id",
            "reason": "Anonymized identifier only",
        },
        {
            "field": "report_date",
            "reason": "Used only to define time windows",
        },
        {
            "field": "product scores and flags",
            "reason": "Would create circular results",
        },
        {
            "field": "raw URLs, domains, queries, and titles",
            "reason": "Privacy and public-safety exclusion",
        },
        {
            "field": "HF_TOKEN",
            "reason": "Credential stored only in Colab Secrets",
        },
        {
            "field": "June 2026 sample",
            "reason": "Reserved as a sealed later test month",
        },
    ]
)

excluded_fields

,field,reason
0,outcome_impressions,Measured after the decision moment
1,outcome_daily_impressions,Future-window measurement
2,outcome_available_days,"Target-window context, not a feature"
3,decline_ratio,Directly defines the proxy label
4,is_declining_proxy,"Target only, never a model feature"
5,client_hash_id,Grouping and validation only
6,content_hash_id,Anonymized identifier only
7,report_date,Used only to define time windows
8,product scores and flags,Would create circular results
9,"raw URLs, domains, queries, and titles",Privacy and public-safety exclusion


In [21]:
print("FINAL FEATURE VECTOR RECEIPT")
print("-" * 50)
print("Rows:", f"{len(X_final):,}")
print("Columns:", X_final.shape[1])
print("Missing values:", int(X_final.isna().sum().sum()))
print("Target positive rate:", round(y_final.mean(), 3))
print("Identifiers in model vector:", identifier_leaks)
print("Leakage fields found:", exact_leaks)
print("Privacy-risk fields found:", privacy_risks)

assert len(X_final) == len(y_final)
assert X_final.isna().sum().sum() == 0
assert len(identifier_leaks) == 0
assert len(exact_leaks) == 0
assert len(privacy_risks) == 0

print("\nFinal feature vector passed all checks.")

FINAL FEATURE VECTOR RECEIPT
--------------------------------------------------
Rows: 76,201
Columns: 10
Missing values: 0
Target positive rate: 0.345
Identifiers in model vector: []
Leakage fields found: []
Privacy-risk fields found: []

Final feature vector passed all checks.


## Results Summary

I built a page-level feature vector using only measurements from March 1–15, 2026.

The vector contains engineered numeric features, including log impressions and CTR, and a one-hot encoded categorical position band.

Numeric missing values were filled with development-slice medians. Missing categorical values were assigned to an `unknown` category.

Client and content identifiers were kept outside the model matrix. They are used only for grouped validation and review.

The automated leakage audit found no future-window, target-derived, product-decision, private, or identifier fields inside the honest feature matrix.

The deliberate leakage experiment showed that adding `decline_ratio` caused an unrealistically large score increase because it directly encoded the target. The field was removed after the test.

The final feature vector is suitable for decision-support experimentation, but it does not prove causal refresh impact or search-engine ranking factors.

## Self-check

- [x] I built the feature vector with executable code.
- [x] I created engineered numeric features.
- [x] I handled a categorical feature with one-hot encoding.
- [x] I showed missing-value handling.
- [x] I explained the meaning of each feature.
- [x] I explained when every feature was available.
- [x] I checked for label-derived fields.
- [x] I checked for future-window fields.
- [x] I checked for product scores and decision flags.
- [x] I checked for private and identifying fields.
- [x] I demonstrated one deliberate leakage trap.
- [x] I removed the leaking field.
- [x] I kept client and content IDs outside the model matrix.
- [x] I included no client names, URLs, domains, or private queries.
- [x] The notebook runs from top to bottom without errors.
- [x] My conclusions use observational and decision-support language.